In [1]:
from pathlib import Path

pdf_dir = Path.cwd()

pdf_files = sorted(pdf_dir.glob("*.pdf"))

print("PDFs found:", len(pdf_files))
for pdf in pdf_files:
    print("-", pdf.name)

PDFs found: 5
- computer_vision_robotics.pdf
- pid_control.pdf
- robot_sensors.pdf
- robotics_reference.pdf
- ros2_basics.pdf


In [4]:
from haystack.components.converters import PyPDFToDocument

converter = PyPDFToDocument()
documents = []

for pdf in pdf_files:
    result = converter.run(sources=[pdf])
    documents.extend(result["documents"])

print("Total documents loaded:", len(documents))

Total documents loaded: 5


In [5]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()
document_store.write_documents(documents)

print("Documents indexed:", document_store.count_documents())

Documents indexed: 5


In [6]:
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

bm25_retriever = InMemoryBM25Retriever(document_store=document_store)

print("BM25 Retriever ready:", bm25_retriever is not None)

BM25 Retriever ready: True


In [7]:
query = "What is computer vision in robotics?"

results = bm25_retriever.run(query=query, top_k=3)

for i, doc in enumerate(results["documents"], start=1):
    print(f"\nResult {i}")
    print("Source:", doc.meta.get("file_path", "Unknown"))
    print("Content:", doc.content[:300] if doc.content else "No content")


Result 1
Source: computer_vision_robotics.pdf
Content: Computer Vision for Robotics
Computer Vision
Computer vision allows robots to extract useful information from images and video for perception and
decision-making.
Image Processing
Common operations include resizing, filtering, color conversion, thresholding, edge detection, and feature
extraction.
O

Result 2
Source: pid_control.pdf
Content: PID Control in Robotics
PID Purpose
A PID controller regulates a system using the error between a desired setpoint and measured output.
Proportional Term
The proportional term produces a response based on the current error.
Integral Term
The integral term accumulates past error and can reduce steady

Result 3
Source: robot_sensors.pdf
Content: Robot Sensors Reference
LiDAR
LiDAR uses laser light to measure distances. It is useful for mapping, obstacle detection, and navigation.
IMU
An IMU measures quantities such as linear acceleration and angular velocity. These measurements help
estimate rob

In [9]:
questions = [
    "What is computer vision in robotics?",
    "What are common image processing operations?",
    "What is the purpose of a PID controller?",
    "What do proportional and integral terms do in PID control?",
    "What does an IMU measure?",
    "How do wheel encoders help robots?",
    "What are the main components of a robot?",
    "What is ROS 2 used for?",
    "How do robot sensors support navigation?",
    "How can robots use computer vision for decision-making?"
]

for i, question in enumerate(questions, 1):
    result = bm25_retriever.run(query=question, top_k=3)
    print(f"\nQ{i}: {question}")
    for doc in result["documents"]:
        print("-", doc.meta.get("file_path", "Unknown"))


Q1: What is computer vision in robotics?
- computer_vision_robotics.pdf
- pid_control.pdf
- robot_sensors.pdf

Q2: What are common image processing operations?
- computer_vision_robotics.pdf
- robot_sensors.pdf
- robotics_reference.pdf

Q3: What is the purpose of a PID controller?
- pid_control.pdf
- robotics_reference.pdf
- ros2_basics.pdf

Q4: What do proportional and integral terms do in PID control?
- pid_control.pdf
- robotics_reference.pdf
- computer_vision_robotics.pdf

Q5: What does an IMU measure?
- robot_sensors.pdf
- robotics_reference.pdf

Q6: How do wheel encoders help robots?
- robot_sensors.pdf
- robotics_reference.pdf
- computer_vision_robotics.pdf

Q7: What are the main components of a robot?
- pid_control.pdf
- robotics_reference.pdf
- ros2_basics.pdf

Q8: What is ROS 2 used for?
- ros2_basics.pdf
- robotics_reference.pdf
- computer_vision_robotics.pdf

Q9: How do robot sensors support navigation?
- robotics_reference.pdf
- robot_sensors.pdf
- ros2_basics.pdf

Q10: H

In [10]:
for i, question in enumerate(questions, 1):
    result = bm25_retriever.run(query=question, top_k=1)
    doc = result["documents"][0]
    print(f"Q{i}: {question}")
    print("Top PDF:", Path(doc.meta.get("file_path", "Unknown")).name)
    print()

Q1: What is computer vision in robotics?
Top PDF: computer_vision_robotics.pdf

Q2: What are common image processing operations?
Top PDF: computer_vision_robotics.pdf

Q3: What is the purpose of a PID controller?
Top PDF: pid_control.pdf

Q4: What do proportional and integral terms do in PID control?
Top PDF: pid_control.pdf

Q5: What does an IMU measure?
Top PDF: robot_sensors.pdf

Q6: How do wheel encoders help robots?
Top PDF: robot_sensors.pdf

Q7: What are the main components of a robot?
Top PDF: pid_control.pdf

Q8: What is ROS 2 used for?
Top PDF: ros2_basics.pdf

Q9: How do robot sensors support navigation?
Top PDF: robotics_reference.pdf

Q10: How can robots use computer vision for decision-making?
Top PDF: computer_vision_robotics.pdf



In [11]:
bm25_evaluation = [
    (1, "computer_vision_robotics.pdf", "Relevant"),
    (2, "computer_vision_robotics.pdf", "Relevant"),
    (3, "pid_control.pdf", "Relevant"),
    (4, "pid_control.pdf", "Relevant"),
    (5, "robot_sensors.pdf", "Relevant"),
    (6, "robot_sensors.pdf", "Relevant"),
    (7, "pid_control.pdf", "Needs review"),
    (8, "ros2_basics.pdf", "Relevant"),
    (9, "robot_sensors.pdf", "Check retrieved result"),
    (10, "computer_vision_robotics.pdf", "Relevant"),
]

for item in bm25_evaluation:
    print(item)

(1, 'computer_vision_robotics.pdf', 'Relevant')
(2, 'computer_vision_robotics.pdf', 'Relevant')
(3, 'pid_control.pdf', 'Relevant')
(4, 'pid_control.pdf', 'Relevant')
(5, 'robot_sensors.pdf', 'Relevant')
(6, 'robot_sensors.pdf', 'Relevant')
(7, 'pid_control.pdf', 'Needs review')
(8, 'ros2_basics.pdf', 'Relevant')
(9, 'robot_sensors.pdf', 'Check retrieved result')
(10, 'computer_vision_robotics.pdf', 'Relevant')


In [13]:
import haystack.components.embedders as embedders

print("Available embedding components:")
print([name for name in dir(embedders) if "Sentence" in name or "Embedder" in name])

Available embedding components:
['AzureOpenAIDocumentEmbedder', 'AzureOpenAITextEmbedder', 'MockDocumentEmbedder', 'MockTextEmbedder', 'OpenAIDocumentEmbedder', 'OpenAITextEmbedder']


In [14]:
import importlib.util

print("sentence_transformers installed:",
      importlib.util.find_spec("sentence_transformers") is not None)

sentence_transformers installed: True


In [17]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

print("Dense retrieval components: OK")

Dense retrieval components: OK


In [18]:
document_embedder = SentenceTransformersDocumentEmbedder()
document_embedder.warm_up()

embedded_documents = document_embedder.run(documents)["documents"]

print("Documents embedded:", len(embedded_documents))
print("Embedding dimension:", len(embedded_documents[0].embedding))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ughan\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Documents embedded: 5
Embedding dimension: 768


In [20]:
from haystack.document_stores.types import DuplicatePolicy

document_store.write_documents(
    embedded_documents,
    policy=DuplicatePolicy.OVERWRITE
)

print("Documents in store:", document_store.count_documents())

Documents in store: 5


In [21]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

dense_retriever = InMemoryEmbeddingRetriever(
    document_store=document_store
)

print("Dense Retriever ready:", dense_retriever is not None)

Dense Retriever ready: True


In [23]:
dense_results = []

for i, question in enumerate(questions, 1):
    query_embedder = SentenceTransformersTextEmbedder()
    query_embedder.warm_up()
    query_embedding = query_embedder.run(text=question)["embedding"]

    result = dense_retriever.run(query_embedding=query_embedding, top_k=3)

    top_pdf = Path(
        result["documents"][0].meta.get("file_path", "Unknown")
    ).name

    dense_results.append((i, question, top_pdf))
    print(f"Q{i}: {question}")
    print("Top PDF:", top_pdf)
    print()
    

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q1: What is computer vision in robotics?
Top PDF: computer_vision_robotics.pdf



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q2: What are common image processing operations?
Top PDF: computer_vision_robotics.pdf



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Q8: What is ROS 2 used for?
Top PDF: ros2_basics.pdf



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q9: How do robot sensors support navigation?
Top PDF: robot_sensors.pdf



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q10: How can robots use computer vision for decision-making?
Top PDF: computer_vision_robotics.pdf



In [24]:
print(f"{'Q':<4} {'BM25 Top PDF':<35} {'Dense Top PDF'}")
print("-" * 85)

for i, question, dense_pdf in dense_results:
    bm25_result = bm25_retriever.run(query=question, top_k=1)
    bm25_pdf = Path(
        bm25_result["documents"][0].meta.get("file_path", "Unknown")
    ).name

    print(f"{i:<4} {bm25_pdf:<35} {dense_pdf}")

Q    BM25 Top PDF                        Dense Top PDF
-------------------------------------------------------------------------------------
1    computer_vision_robotics.pdf        computer_vision_robotics.pdf
2    computer_vision_robotics.pdf        computer_vision_robotics.pdf
3    pid_control.pdf                     pid_control.pdf
4    pid_control.pdf                     pid_control.pdf
5    robot_sensors.pdf                   robot_sensors.pdf
6    robot_sensors.pdf                   robot_sensors.pdf
7    pid_control.pdf                     robotics_reference.pdf
8    ros2_basics.pdf                     ros2_basics.pdf
9    robotics_reference.pdf              robot_sensors.pdf
10   computer_vision_robotics.pdf        computer_vision_robotics.pdf


In [25]:
comparison = []

for i, question, dense_pdf in dense_results:
    bm25_result = bm25_retriever.run(query=question, top_k=1)
    bm25_pdf = Path(
        bm25_result["documents"][0].meta.get("file_path", "Unknown")
    ).name

    comparison.append({
        "question": question,
        "bm25_top_pdf": bm25_pdf,
        "dense_top_pdf": dense_pdf,
        "agree": bm25_pdf == dense_pdf,
    })

agreement = sum(row["agree"] for row in comparison) / len(comparison)

print("Questions evaluated:", len(comparison))
print("Top-result agreement:", f"{agreement:.0%}")
for row in comparison:
    print(row)

Questions evaluated: 10
Top-result agreement: 80%
{'question': 'What is computer vision in robotics?', 'bm25_top_pdf': 'computer_vision_robotics.pdf', 'dense_top_pdf': 'computer_vision_robotics.pdf', 'agree': True}
{'question': 'What are common image processing operations?', 'bm25_top_pdf': 'computer_vision_robotics.pdf', 'dense_top_pdf': 'computer_vision_robotics.pdf', 'agree': True}
{'question': 'What is the purpose of a PID controller?', 'bm25_top_pdf': 'pid_control.pdf', 'dense_top_pdf': 'pid_control.pdf', 'agree': True}
{'question': 'What do proportional and integral terms do in PID control?', 'bm25_top_pdf': 'pid_control.pdf', 'dense_top_pdf': 'pid_control.pdf', 'agree': True}
{'question': 'What does an IMU measure?', 'bm25_top_pdf': 'robot_sensors.pdf', 'dense_top_pdf': 'robot_sensors.pdf', 'agree': True}
{'question': 'How do wheel encoders help robots?', 'bm25_top_pdf': 'robot_sensors.pdf', 'dense_top_pdf': 'robot_sensors.pdf', 'agree': True}
{'question': 'What are the main com

In [27]:
import haystack.components as components

print([name for name in dir(components) if "reader" in name.lower()])
print([name for name in dir(components) if "generator" in name.lower()])

[]
['generators']


In [28]:
import haystack.components

print("Available component modules:")
print(dir(haystack.components))

Available component modules:
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'converters', 'embedders', 'generators', 'retrievers']


In [29]:
import importlib.util

print(
    "Haystack integrations installed:",
    importlib.util.find_spec("haystack_integrations") is not None
)

Haystack integrations installed: True


In [30]:
import pkgutil
import haystack_integrations.components

print("Available integration components:")
print([
    module.name
    for module in pkgutil.iter_modules(
        haystack_integrations.components.__path__
    )
])

Available integration components:
[]


In [31]:
import importlib.util

modules = [
    "haystack_integrations.components.converters",
    "haystack_integrations.components.embedders.sentence_transformers",
    "haystack_integrations.components.retrievers",
    "haystack.components.generators",
]

for module in modules:
    print(module, "->", importlib.util.find_spec(module))

haystack_integrations.components.converters -> None
haystack_integrations.components.embedders.sentence_transformers -> ModuleSpec(name='haystack_integrations.components.embedders.sentence_transformers', loader=<_frozen_importlib_external.SourceFileLoader object at 0x0000023C513849B0>, origin='c:\\Users\\ughan\\CynarisInternship\\AIML_PROJECT\\venv\\Lib\\site-packages\\haystack_integrations\\components\\embedders\\sentence_transformers\\__init__.py', submodule_search_locations=['c:\\Users\\ughan\\CynarisInternship\\AIML_PROJECT\\venv\\Lib\\site-packages\\haystack_integrations\\components\\embedders\\sentence_transformers'])
haystack_integrations.components.retrievers -> None


ValueError: haystack.components.generators.__spec__ is None

In [32]:
import haystack.components.generators as generators

print("Generator module loaded:", generators.__name__)
print("Available names:", dir(generators))

Generator module loaded: haystack.components.generators
Available names: ['OpenAIImageGenerator', '__all__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_exports', '_import_structure', '_name', '_objects', 'openai_image_generator']


In [33]:
import importlib.util

print(
    "Transformers generator integration:",
    importlib.util.find_spec(
        "haystack_integrations.components.generators"
    )
)

Transformers generator integration: ModuleSpec(name='haystack_integrations.components.generators', loader=None, submodule_search_locations=_NamespacePath(['c:\\Users\\ughan\\CynarisInternship\\AIML_PROJECT\\venv\\Lib\\site-packages\\haystack_integrations\\components\\generators']))


In [34]:
import pkgutil
import haystack_integrations.components.generators as generators

print([
    module.name
    for module in pkgutil.iter_modules(generators.__path__)
])

['transformers']


In [36]:
import haystack_integrations.components.generators.transformers as transformers

print("Available components:")
print(dir(transformers))

Available components:
['TransformersChatGenerator', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'chat']


In [37]:
import pkgutil
import haystack

print("Haystack package path:", haystack.__path__)

print("Installed reader-related integrations:")
print([
    module.name
    for module in pkgutil.iter_modules(
        haystack_integrations.__path__
    )
    if "reader" in module.name.lower()
])

Haystack package path: ['c:\\Users\\ughan\\CynarisInternship\\AIML_PROJECT\\venv\\Lib\\site-packages\\haystack']
Installed reader-related integrations:
[]


In [38]:
import haystack_integrations

In [39]:
import pandas as pd

comparison_df = pd.DataFrame(comparison)
comparison_df.to_csv(
    "W8D3_retrieval_comparison.csv",
    index=False
)

print("Saved:", "W8D3_retrieval_comparison.csv")
print("Questions evaluated:", len(comparison_df))
print("Top-result agreement:", f"{comparison_df['agree'].mean():.0%}")

Saved: W8D3_retrieval_comparison.csv
Questions evaluated: 10
Top-result agreement: 80%


In [40]:
from pathlib import Path

output_file = Path("W8D3_retrieval_comparison.csv")

print("File exists:", output_file.exists())
print("File size:", output_file.stat().st_size, "bytes")

File exists: True
File size: 932 bytes
